In [1]:
#!pip install datasets evaluate transformers[sentencepiece]
#!apt install git-lfs

In [2]:
!git config --global user.email "sujatkhan24@gmail.com"
!git config --global user.name "sak-07"

In [3]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [4]:
#!pip install requests

In [5]:
import requests

url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url)

In [6]:
response.status_code

200

In [7]:
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/8206',
  'repository_url': 'https://api.github.com/repos/huggingface/datasets',
  'labels_url': 'https://api.github.com/repos/huggingface/datasets/issues/8206/labels{/name}',
  'comments_url': 'https://api.github.com/repos/huggingface/datasets/issues/8206/comments',
  'events_url': 'https://api.github.com/repos/huggingface/datasets/issues/8206/events',
  'html_url': 'https://github.com/huggingface/datasets/pull/8206',
  'id': 4463863740,
  'node_id': 'PR_kwDODunzps7cZASu',
  'number': 8206,
  'title': 'Fix duplicate keyword arguments',
  'user': {'login': 'AtreoP',
   'id': 97874269,
   'node_id': 'U_kgDOBdVxXQ',
   'avatar_url': 'https://avatars.githubusercontent.com/u/97874269?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/AtreoP',
   'html_url': 'https://github.com/AtreoP',
   'followers_url': 'https://api.github.com/users/AtreoP/followers',
   'following_url': 'https://api.github.com/users/Atreo

In [ ]:
GITHUB_TOKEN = ''  # Copy your GitHub token here
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

In [9]:
import time
import math
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm


def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=5_000,
    rate_limit=5_001,
    issues_path=Path("."),
):
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    batch = []
    all_issues = []
    per_page = 100  # Number of issues to return per page
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    for page in tqdm(range(num_pages)):
        # Query with state=all to get both open and closed issues
        query = f"issues?page={page}&per_page={per_page}&state=all"
        issues = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)
        batch.extend(issues.json())

        if len(batch) > rate_limit and len(all_issues) < num_issues:
            all_issues.extend(batch)
            batch = []  # Flush batch for next time period
            print(f"Reached GitHub rate limit. Sleeping for one hour ...")
            time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    df = pd.DataFrame.from_records(all_issues)
    df.to_json(f"{issues_path}/{repo}-issues.jsonl", orient="records", lines=True)
    print(
        f"Downloaded all the issues for {repo}! Dataset stored at {issues_path}/{repo}-issues.jsonl"
    )

In [10]:
# Depending on your internet connection, this can take several minutes to run...
fetch_issues()

  0%|          | 0/50 [00:00<?, ?it/s]

Downloaded all the issues for datasets! Dataset stored at ./datasets-issues.jsonl


In [11]:
from datasets import load_dataset
issues_dataset = load_dataset("json", data_files="datasets-issues.jsonl", split="train")
issues_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment'],
    num_rows: 5000
})

In [12]:
sample = issues_dataset.shuffle(seed=666).select(range(3))

# Print out the URL and pull request entries
for url, pr in zip(sample["html_url"], sample["pull_request"]):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}\n")

>> URL: https://github.com/huggingface/datasets/pull/7476
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/7476', 'html_url': 'https://github.com/huggingface/datasets/pull/7476', 'diff_url': 'https://github.com/huggingface/datasets/pull/7476.diff', 'patch_url': 'https://github.com/huggingface/datasets/pull/7476.patch', 'merged_at': datetime.datetime(2025, 3, 25, 15, 45)}

>> URL: https://github.com/huggingface/datasets/pull/4914
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/4914', 'html_url': 'https://github.com/huggingface/datasets/pull/4914', 'diff_url': 'https://github.com/huggingface/datasets/pull/4914.diff', 'patch_url': 'https://github.com/huggingface/datasets/pull/4914.patch', 'merged_at': datetime.datetime(2022, 8, 30, 11, 14, 15)}

>> URL: https://github.com/huggingface/datasets/issues/7310
>> Pull request: None



In [13]:
issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": False if x["pull_request"] is None else True}
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [14]:
issue_number = 2792
url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
response = requests.get(url, headers=headers)
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/897594128',
  'html_url': 'https://github.com/huggingface/datasets/pull/2792#issuecomment-897594128',
  'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/2792',
  'id': 897594128,
  'node_id': 'IC_kwDODunzps41gDMQ',
  'user': {'login': 'bhavitvyamalik',
   'id': 19718818,
   'node_id': 'MDQ6VXNlcjE5NzE4ODE4',
   'avatar_url': 'https://avatars.githubusercontent.com/u/19718818?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/bhavitvyamalik',
   'html_url': 'https://github.com/bhavitvyamalik',
   'followers_url': 'https://api.github.com/users/bhavitvyamalik/followers',
   'following_url': 'https://api.github.com/users/bhavitvyamalik/following{/other_user}',
   'gists_url': 'https://api.github.com/users/bhavitvyamalik/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/bhavitvyamalik/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/

In [21]:
def get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    response = requests.get(url, headers=headers)
    data = response.json()
    if isinstance(data, dict):
        return []
    return [r["body"] for r in data]


# Test our function works as expected
get_comments(2792)

[]

In [24]:
# Depending on your internet connection, this can take a few minutes...
issues_with_comments_dataset = issues_dataset.map(
    lambda x: {"comments": get_comments(x["number"])}
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [25]:
from huggingface_hub import notebook_login

notebook_login()

In [26]:
issues_with_comments_dataset.push_to_hub("github-issues")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  12%|#1        | 1.17MB / 10.0MB            

CommitInfo(commit_url='https://huggingface.co/datasets/SAK-07/github-issues/commit/b060680d43b84c6cb1c75cd6eeab84e4ee9ed942', commit_message='Upload dataset', commit_description='', oid='b060680d43b84c6cb1c75cd6eeab84e4ee9ed942', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/SAK-07/github-issues', endpoint='https://huggingface.co', repo_type='dataset', repo_id='SAK-07/github-issues'), pr_revision=None, pr_num=None)

In [27]:
remote_dataset = load_dataset("SAK-07/github-issues", split="train")
remote_dataset

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 5000
})